# Trading Engine Details

The [lecture](../CHEME-5660-L15a-Lecture-Testing-Trading-Strategies-Fall-2026.ipynb) uses the engine to backtest and stress-test one strategy. Here we look at the estimates, the preference score, and the optional market hedge that enter the engine's decisions.

___

## Fixed and Updated Estimates

The engine starts from a 2016 to 2024 fit of the single index model. It can keep the estimates fixed or update each stock's baseline and beta at each close. Beta measures a stock's sensitivity to market growth.

The update uses exponentially weighted least squares (EWLS). A day's weight halves every 63 trading days. The starting fit carries the weight of 63 new trading days, and its weight decays at the same rate.

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="../figs/Fig-L15a-EWLS-JPM.svg" width="740" style="max-width:100%; height:auto;" alt="The market sensitivity of JPM over 2025 and 2026. A dashed line marks the fixed estimate from the 2016 to 2024 fit, 1.12. The blue updated estimate, with one update per trading day, starts at 1.12, stays near 1.0 through 2025, and drops to about 0.7 in mid-2026. Late in 2026, JPM moves less with the market than the fixed estimate says."/>
    </center>
</div>

In the figure, the fixed beta for JPM is 1.12 and the updated estimate ends at 0.73. The [updating example](../CHEME-5660-L15a-Example-Updating-SIM-Gamma-Fall-2026.ipynb) computes these estimates and follows how they change the basket. The [EWLS derivation](ewls/CHEME-5660-L15a-Derivation-EWLS-Update-Fall-2026.ipynb) gives the weighted fit and its daily update.

___

## The Preference Score

The daily market state $\xi_{t}$ measures the gap between the 21-day and 63-day exponential moving averages of the `SPY` price. With $\bar{S}^{\mathrm{short}}_{t}$ and $\bar{S}^{\mathrm{long}}_{t}$ denoting those averages, the state is given by:

$$
\xi_{t} = -G\left(\frac{\bar{S}^{\mathrm{short}}_{t}}{\bar{S}^{\mathrm{long}}_{t}}-1\right),
$$

where the gain $G>0$ scales $\xi_t$ so that $|\xi_t|\leq 1$ on 99% of training days. The state is positive when the short average lies below the long average.

The single index model gives the expected growth rate $\tilde{g}_{i,t}$ (1/yr) of stock $i$ from its baseline $\alpha_i(t)$ (1/yr), its market sensitivity $\beta_i(t)$, and the 252-day exponential moving average $\tilde{g}_{M,t}$ of market growth (1/yr). The preference score $\gamma_i(t)$ for this stock is given by:

$$
\begin{aligned}
\tilde{g}_{i,t} &= \alpha_i(t)+\beta_i(t)\,\tilde{g}_{M,t},\\[4pt]
\gamma_i(t) &= \tanh\left(h\,\frac{\tilde{g}_{i,t}}{\max\!\left(|\beta_i(t)|,\,\beta_{\mathrm{floor}}\right)^{\xi_t}}\right),
\end{aligned}
$$

where $h=1$ year makes the argument dimensionless and $\beta_{\mathrm{floor}}=0.1$ keeps the denominator positive. Among stocks with positive expected growth, a positive market state favors lower absolute beta at the same expected growth rate.

The score updates as new prices arrive. The picker uses today's scores when a trigger calls for a new basket. The figure shows scheduled picks and one kind of market turn.

<div>
    <center>
        <img class="course-diagram" src="../figs/Fig-L15a-Gamma-Picks.svg" width="860" style="max-width:100%; height:auto;" alt="One time axis with three rows. The top row is a path of the market state xi t, shaded blue where the market is rising and red where it is falling. The middle row is a line with a dot for every trading day, when alpha, beta, and gamma update. The bottom row shows the preference baskets held, numbered 1 to 7, each from one pick day to the next. Most baskets last 21 trading days. A brace over basket 2 reads: 15 trading days, cut short by the market turn. Where xi t crosses zero, an orange dashed line marks a market turn: the method picks a new basket, and the 21-day count restarts."/>
    </center>
</div>

___

## The Market Hedge

With the optional hedge on, the engine caps the portfolio's market sensitivity at each new decision. Let $w_i(t)$ be the weight of holding $i$ as a fraction of wealth, with a negative weight for a short. The portfolio's market sensitivity $\beta_P(t)$ must satisfy:

$$
\beta_P(t)=\sum_i w_i(t)\,\beta_i(t)\leq \max\!\left(1-\xi_t,\,-0.5\right).
$$

Cash has zero beta. The cap falls as the market state rises.

<div>
    <center>
        <img class="course-diagram" src="../figs/Fig-L15a-Market-Hedge.svg" width="720" style="max-width:100%; height:auto;" alt="A line shows the limit on the market sensitivity of the portfolio, beta P of t, against the market state xi t. The limit falls from 2 at xi t equals minus 1 to 1 at xi t equals 0, the market sensitivity of SPY. It reaches 0 at xi t equals 1, the market sensitivity of betting against beta. Below zero, a red area marks where the method sells SPY short, and the line flattens at a floor of minus 0.5. Arrows from the line point down: beta P of t stays on or below the line. A gray band marks where the absolute value of xi t is at most 1, which holds on 99 percent of training days."/>
    </center>
</div>

- When $\xi_t<0$, the cap is above one. The hedge allows the portfolio to be more sensitive to the market than `SPY`.
- As $\xi_t$ rises from zero to one, the cap drops from one to zero. The engine can put more weight in low-beta stocks or cash. It can also add up to five stocks with zero or negative beta from its hedge pool.
- When $\xi_t>1$, the engine shorts `SPY`. A short worth 20% of the portfolio's value lowers portfolio beta by 0.2. The floor of −0.5 caps the short at half of the portfolio's value.

___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team.

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___